# Inspect Gadget Circuits, Checks & Readouts

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

What are the physical components inside a `qodec.Gadget`, and how do measurement readouts and input-to-output parity checks compose across rounds?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Inspect the internal attributes of a `qodec.Gadget`: `implements`, `circuit`, `in_`, `out`, `readouts`, and `checks`.
- Trace how `qodec` expresses parity checks as input-to-output stabilizer boundary equations rather than monolithic multi-round detector indices.
- Render and inspect gadget circuits and individual `InstructionCall` steps.

## Before you begin

This course requires `qdk[ec]>=1.33.0`, `qodec>=0.2.0`, `paulimer`, `binar`, and `ipykernel`. Run the cell below to verify your active Python kernel and environment.

In [ ]:
# If packages are missing, first select a dedicated Python environment/kernel,
# then uncomment the next line and run this cell again.
# %pip install -r ../requirements.txt

from _unit import check_env

check_env()

## What Inside a `qodec.Gadget` Connects Physical Circuits to Logical Semantics?

A `qodec.Gadget` implements one logical `Instruction` from Layer $L$ using a circuit written in the primitives of Layer $L+1$. Every gadget has four key components:
1. **`gadget.implements`**: The target logical `Instruction` (defining the objective instruction actions and optional flags).
2. **`gadget.circuit`**: The physical implementation circuit (e.g., in `format: stim` or `format: qasm`). Qubit indices are mapped sequentially across the input/output code blocks (`0..3` for the first $C_4$ block, `4..7` for a second $C_4$ block or ancilla qubits).
3. **`gadget.readouts`**: Parity combinations of classical measurement outcomes (`circuit.readouts[i]`) that reconstruct the logical observables (`observe[0]`, `observe[1]`) or rejection flags (`reject`).
4. **`gadget.checks`**: Parity combinations of input stabilizers (`in[0].s[0]`), output stabilizers (`out[0].s[0]`), and classical measurement bits (`circuit.readouts[i]`) that must evaluate to $+1$ (even parity) in every noiseless execution (**parity checks**).
   - *QEC Researcher Note (Why `qodec` expresses checks on block boundaries):* Rather than tying checks to fixed multi-round measurement offsets, `qodec` expresses each check as a local parity equation over input stabilizers, output stabilizers, and gadget measurements. Because `qodec` gadgets are modular operations that can be called in *any* order by a Q# or OpenQASM program, `qodec` expresses each check as a **boundary stabilizer relation** between the entering block (`in[0]`), exiting block (`out[0]`), and local circuit measurements (`circuit.readouts[i]`). At simulation/decoding time, `qdk.simulation` and `deq` automatically stitch `out[0]` of one gadget to `in[0]` of the next gadget to form multi-gadget parity checks!

Let's inspect three contrasting gadgets from `protocol.layers[0].gadgets`:
- **`measure_z_all`**: Destructive transversal measurement (`M 0 1 2 3`).
- **`cx_all`**: Two-block transversal CNOT (`CX 0 4`, `CX 1 5`, `CX 2 6`, `CX 3 7`).
- **`syndrome`**: Bare-ancilla CSS syndrome extraction using ancilla qubits `4` ($X$-stabilizer) and `5` ($Z$-stabilizer).

In [2]:
import qodec
import qdk.ec as ec
from _unit import simulation

# Inspect `measure_z_all`, `cx_all`, and `syndrome` Gadgets
from _unit import simulation
simulation.ensure_course_state(globals(), stage="baseline")
gadgets = protocol.layers[0].gadgets

for name in ["measure_z_all", "cx_all", "syndrome"]:
    g = gadgets[name]
    print(f"=== Gadget: {name} ===")
    print(f"Inputs : {list(g.inputs)} | Outputs : {list(g.outputs)}")
    print("Circuit:")
    for ln in g.circuit.source.strip().splitlines():
        print(f"  {ln}")
    print(f"Readouts : {g.readouts}")
    print(f"Checks   : {g.checks}\n")

=== Gadget: measure_z_all ===
Inputs : [Encoding(code="C4")] | Outputs : []
Circuit:
  M 0 1 2 3
Readouts : (Readout(position=0, name=None, is_flag=False, equation=['in[0].z[0]', 'circuit.readouts[0]', 'circuit.readouts[2]']), Readout(position=1, name=None, is_flag=False, equation=['in[0].z[1]', 'circuit.readouts[0]', 'circuit.readouts[1]']))
Checks   : ((Reference('circuit.readouts[0]'), Reference('circuit.readouts[1]'), Reference('circuit.readouts[2]'), Reference('circuit.readouts[3]'), Reference('in[0].stabilizers[1]')),)

=== Gadget: cx_all ===
Inputs : [Encoding(code="C4"), Encoding(code="C4")] | Outputs : [Encoding(code="C4"), Encoding(code="C4")]
Circuit:
  CX 0 4
  CX 1 5
  CX 2 6
  CX 3 7
Readouts : ()
Checks   : ((Reference('out[0].stabilizers[0]'), Reference('in[0].stabilizers[0]'), Reference('in[1].stabilizers[0]')), (Reference('out[0].stabilizers[1]'), Reference('in[0].stabilizers[1]')), (Reference('out[1].stabilizers[0]'), Reference('in[1].stabilizers[0]')), (Reference('o

## Visualizing Gadget Circuits & Indexing Circuit Calls

When analyzing where faults can occur inside a gadget, QDK-EC indexes each elementary gate call in `gadget.circuit.calls()` from `0` to `N-1`. Note that multi-target Stim lines like `R 4 5` or `M 4 5` expand into individual single-qubit calls (`0: R 4`, `1: R 5`, ..., `11: M 4`, `12: M 5`). Let's inspect the call table and render the circuit diagram with `stim.Circuit(syndrome_gadget.circuit.source).diagram("timeline-svg-html")`.

In [3]:
# Inspect Individual Circuit Calls & Diagram for `syndrome`
from _unit import simulation
simulation.ensure_course_state(globals(), stage="baseline")
import stim

gadgets = protocol.layers[0].gadgets
syndrome_gadget = gadgets["syndrome"]
print("Indexed elementary calls in baseline `syndrome` gadget:")
for idx, call in enumerate(syndrome_gadget.circuit.calls()):
    print(f"  call[{idx:02d}] : {call.mnemonic} {call.operands}")

stim.Circuit(syndrome_gadget.circuit.source).diagram("timeline-svg-html")

Indexed elementary calls in baseline `syndrome` gadget:
  call[00] : R [4]
  call[01] : H [4]
  call[02] : CX [4, 0]
  call[03] : CX [4, 1]
  call[04] : CX [4, 2]
  call[05] : CX [4, 3]
  call[06] : H [4]
  call[07] : R [5]
  call[08] : H [5]
  call[09] : CZ [5, 0]
  call[10] : CZ [5, 1]
  call[11] : CZ [5, 2]
  call[12] : CZ [5, 3]
  call[13] : H [5]
  call[14] : M [4]
  call[15] : M [5]


<svg viewBox="0 0 992 480"  version="1.1" xmlns="http://www.w3.org/2000/svg">
<g id="qubit_lines">
<path id="qubit_line:0" d="M64,64 L960,64 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="64">q0</text>
<path id="qubit_line:1" d="M64,128 L960,128 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="128">q1</text>
<path id="qubit_line:2" d="M64,192 L960,192 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="192">q2</text>
<path id="qubit_line:3" d="M64,256 L960,256 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="256">q3</text>
<path id="qubit_line:4" d="M64,320 L960,320 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="320">q4</text>
<path id="qubit_line:5" d="M64,384 L960,384 " stroke="black"/>
<text dominant-baseline="central" text-anchor="end" font-family="monospace" font-size="12" x="64" y="384">q5</text>
</g>
<rect x="80" y="304" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="96" y="320" fill="white">R</text>
<rect x="144" y="304" width="32" height="32" stroke="black" fill="white"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="160" y="320">H</text>
<path d="M224,64 L224,320 " stroke="black"/>
<circle cx="224" cy="320" r="12" stroke="none" fill="black"/>
<circle cx="224" cy="64" r="12" stroke="black" fill="white"/>
<path d="M212,64 L236,64 M224,52 L224,76 " stroke="black"/>
<path d="M288,128 L288,320 " stroke="black"/>
<circle cx="288" cy="320" r="12" stroke="none" fill="black"/>
<circle cx="288" cy="128" r="12" stroke="black" fill="white"/>
<path d="M276,128 L300,128 M288,116 L288,140 " stroke="black"/>
<path d="M352,192 L352,320 " stroke="black"/>
<circle cx="352" cy="320" r="12" stroke="none" fill="black"/>
<circle cx="352" cy="192" r="12" stroke="black" fill="white"/>
<path d="M340,192 L364,192 M352,180 L352,204 " stroke="black"/>
<path d="M416,256 L416,320 " stroke="black"/>
<circle cx="416" cy="320" r="12" stroke="none" fill="black"/>
<circle cx="416" cy="256" r="12" stroke="black" fill="white"/>
<path d="M404,256 L428,256 M416,244 L416,268 " stroke="black"/>
<rect x="464" y="304" width="32" height="32" stroke="black" fill="white"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="480" y="320">H</text>
<rect x="464" y="368" width="32" height="32" stroke="black" fill="black"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="480" y="384" fill="white">R</text>
<rect x="528" y="368" width="32" height="32" stroke="black" fill="white"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="544" y="384">H</text>
<path d="M608,64 L608,384 " stroke="black"/>
<circle cx="608" cy="384" r="12" stroke="none" fill="black"/>
<circle cx="608" cy="64" r="12" stroke="none" fill="black"/>
<path d="M672,128 L672,384 " stroke="black"/>
<circle cx="672" cy="384" r="12" stroke="none" fill="black"/>
<circle cx="672" cy="128" r="12" stroke="none" fill="black"/>
<path d="M736,192 L736,384 " stroke="black"/>
<circle cx="736" cy="384" r="12" stroke="none" fill="black"/>
<circle cx="736" cy="192" r="12" stroke="none" fill="black"/>
<path d="M800,256 L800,384 " stroke="black"/>
<circle cx="800" cy="384" r="12" stroke="none" fill="black"/>
<circle cx="800" cy="256" r="12" stroke="none" fill="black"/>
<rect x="848" y="368" width="32" height="32" stroke="black" fill="white"/>
<text dominant-baseline="central" text-anchor="middle" font-family="monospace" font-size="30" x="864" y="384">H</text>
<rect x="848" y="304" wid

## Knowledge check

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;In `gadgets['measure_z_all']`, why are there two entries in `readouts` and one entry in `checks`?</summary>

<div style="padding:0.1em 1em;">

Because $C_4$ encodes $k=2$ logical qubits (`Z_0 = Z_0 Z_2` and `Z_1 = Z_0 Z_1`), their values are reconstructed from `readouts[0] ^ readouts[2]` and `readouts[0] ^ readouts[1]`. The remaining independent parity combination `readouts[0] ^ readouts[1] ^ readouts[2] ^ readouts[3]` measures the $Z$ stabilizer $Z_0 Z_1 Z_2 Z_3$, which must match the input block's stabilizer `inputs[0]:Z_0 Z_1 Z_2 Z_3`—forming a syndrome check.

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;In `gadgets['syndrome']`, why does each check in `g.checks` include both `inputs[0]:...`, `outputs[0]:...`, and `circuit.readouts[...]`?</summary>

<div style="padding:0.1em 1em;">

A syndrome extraction check (parity check) compares the stabilizer value entering the gadget (`inputs[0]`) with the stabilizer value leaving the gadget (`outputs[0]`) and the ancilla measurement bit (`circuit.readouts[i]`). In a noiseless run, the product of the input stabilizer, output stabilizer, and ancilla measurement parity is deterministically $+1$.

</div>

</details></div>

## Exercise: Count Two-Qubit Entangling Calls in Baseline Syndrome Extraction

Iterate over the elementary circuit calls of the baseline `syndrome` gadget (`syn.circuit.calls()`) and return the total number of two-qubit entangling gate calls (`call.mnemonic in ('CX', 'CZ')`).

In [ ]:
from _unit import exercise, simulation


@exercise
def syndrome_cnot_count() -> int:
    ns: dict = {}
    simulation.ensure_course_state(ns, 'baseline')
    syn = ns['protocol'].layers[0].gadgets['syndrome']
    # Count how many calls in syn.circuit.calls() have call.mnemonic in ('CX', 'CZ')
    return ...

**Hint**

Use `sum(1 for call in syn.circuit.calls() if call.mnemonic in ('CX', 'CZ'))`.

In [ ]:
from _unit import exercise, simulation


@exercise
def syndrome_cnot_count() -> int:
    ns: dict = {}
    simulation.ensure_course_state(ns, 'baseline')
    syn = ns['protocol'].layers[0].gadgets['syndrome']
    return sum(1 for call in syn.circuit.calls() if call.mnemonic in ('CX', 'CZ'))

Measuring the weight-4 $X$ stabilizer ($X_0 X_1 X_2 X_3$) onto ancilla `4` uses `4` `CX` gates, and measuring the weight-4 $Z$ stabilizer ($Z_0 Z_1 Z_2 Z_3$) onto ancilla `5` uses `4` `CZ` gates, giving `8` two-qubit entangling gate calls in total.

---

### 🤖 COPILOT INSTRUCTOR BRIEF — Module 3
- **Learning Objectives:**
  1. Inspect `gadget.circuit.source`, `gadget.readouts`, and `gadget.checks` across destructive measurement, unitary two-block gate, and syndrome extraction gadgets.
  2. Understand how `circuit.readouts[i]` maps to the $i$-th measurement operation in `gadget.circuit` (0-indexed).
  3. Trace why `measure_z_all` has two logical readouts (`m0 ^ m2` and `m0 ^ m1`) plus one check (`m0 ^ m1 ^ m2 ^ m3`).
- **Socratic Prompt:** Ask the learner: *"In `measure_z_all`, we measure all 4 physical qubits in the Z basis, getting 4 classical bits `readouts[0..3]`. Two combinations give the logical observables $\bar{Z}_0 = Z_0 Z_2$ and $\bar{Z}_1 = Z_0 Z_1$. What is the 4-bit parity `readouts[0] ^ readouts[1] ^ readouts[2] ^ readouts[3]` measuring, and why is it listed in `gadget.checks`?"*

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../02-instruction-sets/instruction_sets.workbook.ipynb" title="Previous unit: Build a Baseline Qodec & Instruction Set" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Build a Baseline Qodec & Instruction Set</span></span></a><a href="../04-validate-and-audit/validate_and_audit.workbook.ipynb" title="Next unit: Validate & Fill Gadgets with Audit and GadgetProfile" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Validate & Fill Gadgets with Audit and GadgetProfile</span></span><span aria-hidden="true">&#8250;</span></a></div>